# LeetCode 1469 — Find All The Lonely Nodes bằng Python

**Link đề:** https://leetcode.com/problems/find-all-the-lonely-nodes/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def getLonelyNodes(self, root: Optional[TreeNode]) -> List[int]:
        ...
```

### Mục tiêu học
1. Tự định nghĩa `TreeNode`, **dựng cây từ list level-order** kiểu LeetCode (`[1,2,3,None,4]`) và **in cây** để nhìn.
2. Hiểu "nút cô đơn" (lonely node) và vì sao nên **đứng ở nút cha** để quyết định (cha biết mình có mấy con).
3. Đi qua 5 lời giải: bảng `parent` (trực giác) → **DFS đệ quy** → DFS bằng **stack** → **BFS** bằng `deque` → đệ quy trả về list (kiểu hàm).
4. Trace call stack / hàng đợi từng bước, chứng minh, phân tích Time/Space (kể cả độ sâu đệ quy).
5. Biến thể: đếm nút lá, nút có đủ 2 con, *Sum of Left Leaves*, nút cô đơn theo từng tầng.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Trong cây nhị phân, một nút được gọi là **cô đơn** (lonely) nếu nó là **con duy nhất** của nút cha. Nút gốc **không** cô đơn vì không có cha.

Cho `root` của một cây nhị phân, trả về list **giá trị** của mọi nút cô đơn. Thứ tự **tùy ý**.

| Ví dụ | `root` (level-order) | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,null,4]` | `[4]` | 4 là con duy nhất của 2; 2 và 3 là anh em nên không cô đơn |
| 2 | `[7,1,4,6,null,5,3,null,null,null,null,null,2]` | `[6,2]` | 6 là con duy nhất của 1; 2 là con duy nhất của 3 |
| 3 | `[11,99,88,77,null,null,66,55,null,null,44,33,null,null,22]` | `[77,55,33,66,44,22]` | mọi nút trừ gốc, 99, 88 đều là con một |

**Constraints** (theo đề gốc)
- Số nút trong cây trong khoảng `[1, 1000]`.
- `1 <= Node.val <= 10^6`.

Câu hỏi gợi mở:
- Đứng ở một nút, làm sao biết **nó** có cô đơn không? (Nút con không có con trỏ ngược về cha!)
- Đứng ở nút **cha**, câu hỏi trở nên dễ thế nào?

## 2. Nền tảng: cây nhị phân trong Python

### 2.1. Class `TreeNode`

Mỗi nút có `val` và hai con trỏ `left`, `right` (có thể là `None`). `__repr__` giúp in nút cho dễ đọc.

In [1]:
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

# Dựng tay cây ví dụ 1:      1
#                          /   \
#                         2     3
#                          \
#                           4
root = TreeNode(1, TreeNode(2, None, TreeNode(4)), TreeNode(3))
print(root, root.left, root.right, root.left.right)
print("2 có con trái?", root.left.left is not None, "| con phải?", root.left.right is not None)

TreeNode(1) TreeNode(2) TreeNode(3) TreeNode(4)
2 có con trái? False | con phải? True


`x is None` / `x is not None` là cách chuẩn để kiểm tra con trỏ rỗng (so sánh **danh tính**, nhanh và rõ hơn `== None`).

### 2.2. Dựng cây từ list level-order (định dạng LeetCode)

LeetCode ghi cây theo **từng tầng, trái sang phải**; `None` (`null`) đánh dấu chỗ trống. Các con của một chỗ `None` **không được ghi**. Ta dùng hàng đợi để gắn con lần lượt cho từng nút.

In [2]:
from collections import deque

def build_tree(values):
    if not values or values[0] is None:
        return None
    it = iter(values)
    root = TreeNode(next(it))
    q = deque([root])
    for v_left in it:
        node = q.popleft()
        if v_left is not None:
            node.left = TreeNode(v_left); q.append(node.left)
        v_right = next(it, None)          # có thể đã hết list
        if v_right is not None:
            node.right = TreeNode(v_right); q.append(node.right)
    return root

def to_level_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

ex1 = [1, 2, 3, None, 4]
ex2 = [7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2]
ex3 = [11, 99, 88, 77, None, None, 66, 55, None, None, 44, 33, None, None, 22]
for vals in [ex1, ex2, ex3]:
    assert to_level_list(build_tree(vals)) == vals
print("build_tree / to_level_list khớp cả 3 ví dụ")

build_tree / to_level_list khớp cả 3 ví dụ


Công cụ mới trong cell:
- `deque` (hàng đợi hai đầu): `append` thêm vào cuối, `popleft` lấy ở đầu — cả hai `O(1)`. (List `pop(0)` tốn `O(n)`.)
- `iter(values)` tạo **iterator**; `next(it)` lấy phần tử kế; `next(it, None)` trả `None` nếu hết thay vì báo lỗi.
- `to_level_list` làm ngược lại để **kiểm chứng** hàm dựng.

### 2.3. In cây theo dạng nằm ngang

Hàm đệ quy: in cây con phải ở trên, nút ở giữa, cây con trái ở dưới — nghiêng đầu sang trái là thấy cây.

In [3]:
def print_tree(node, indent=0):
    if node is None:
        return
    print_tree(node.right, indent + 6)
    print(" " * indent + f"{node.val}")
    print_tree(node.left, indent + 6)

print_tree(build_tree(ex2))

                  2
            3
      4
            5
7
      1
            6


### 2.4. Đệ quy trên cây — ý tưởng cốt lõi

Một hàm đệ quy trên cây có dạng:
1. **Trường hợp cơ sở:** `node is None` → không làm gì (hoặc trả giá trị trung tính).
2. **Xử lý nút hiện tại.**
3. **Gọi đệ quy** cho `node.left` và `node.right`.

Ví dụ nhỏ: đếm số nút.

In [4]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

print(count_nodes(build_tree(ex1)), count_nodes(build_tree(ex2)), count_nodes(build_tree(ex3)))

4 7 9


### 2.5. "Có đúng một con" — phép XOR trên `bool`

Nút có đúng một con ⇔ `(left is None) != (right is None)`. Với `bool`, `!=` chính là **XOR** (khác nhau). Bảng chân trị:

In [5]:
for has_left in [False, True]:
    for has_right in [False, True]:
        print(f"có trái={has_left!s:5} có phải={has_right!s:5} → đúng 1 con: {has_left != has_right}")

có trái=False có phải=False → đúng 1 con: False
có trái=False có phải=True  → đúng 1 con: True
có trái=True  có phải=False → đúng 1 con: True
có trái=True  có phải=True  → đúng 1 con: False


## 3. Cách 1 — Bảng `parent` (trực giác: "hỏi xem cha có mấy con")

Định nghĩa nói về **cha**, nên cách đầu tiên ta nghĩ: với **mỗi nút**, tìm cha của nó, đếm số con của cha. Vì `TreeNode` không có con trỏ về cha, ta duyệt một lượt để xây `parent[nút] = cha`.

In [6]:
def lonely_parent_map(root):
    parent = {root: None}
    order = [root]
    for node in order:                       # duyệt BFS bằng list đang lớn dần
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                order.append(child)
    result = []
    for node in order:
        p = parent[node]
        if p is None:
            continue                         # gốc không cô đơn
        num_children = (p.left is not None) + (p.right is not None)
        if num_children == 1:
            result.append(node.val)
    return result

print(lonely_parent_map(build_tree(ex1)))
print(lonely_parent_map(build_tree(ex2)))
print(lonely_parent_map(build_tree(ex3)))

[4]
[6, 2]
[77, 66, 55, 44, 33, 22]


Ghi chú kỹ thuật:
- Đối tượng `TreeNode` dùng được làm **key của dict** (băm theo danh tính đối tượng) — hai nút cùng giá trị vẫn là hai key khác nhau. ✓
- `for node in order` trong khi `order.append(...)`: Python duyệt list theo chỉ số nên sẽ duyệt cả phần tử mới thêm — một cách BFS gọn (không xóa phần tử).
- `True + True = 2`: cộng `bool` để đếm số con.

### 3.1. Vấn đề của cách 1

- Đúng, `O(n)`, nhưng tốn **thêm** dict `parent` và list `order` (`O(n)` bộ nhớ) và hai lượt duyệt.
- Mỗi nút cha bị "hỏi" lại một lần cho **mỗi** con của nó.

## 4. Insight — để **cha** quyết định

Nút con không biết cha, nhưng **cha biết hết con của mình**. Đứng ở nút `node`:
- Có con trái, **không** có con phải → con trái cô đơn.
- Có con phải, **không** có con trái → con phải cô đơn.
- Có đủ hai con hoặc không con nào → không ai cô đơn.

Vậy chỉ cần **một lần duyệt**, tại mỗi nút kiểm tra 2 điều kiện — không cần bảng `parent`. Thử tay ở vài nút của ví dụ 2:

In [7]:
r2 = build_tree(ex2)
for node in [r2, r2.left, r2.right, r2.right.right]:
    L, R = node.left, node.right
    if L is not None and R is None:
        verdict = f"con trái {L.val} cô đơn"
    elif R is not None and L is None:
        verdict = f"con phải {R.val} cô đơn"
    else:
        verdict = "không có con cô đơn"
    print(f"nút {node.val}: trái={L}, phải={R} → {verdict}")

nút 7: trái=TreeNode(1), phải=TreeNode(4) → không có con cô đơn
nút 1: trái=TreeNode(6), phải=None → con trái 6 cô đơn
nút 4: trái=TreeNode(5), phải=TreeNode(3) → không có con cô đơn
nút 3: trái=None, phải=TreeNode(2) → con phải 2 cô đơn


## 5. Cách 2 (lời giải chính) — DFS đệ quy

### 5.1. Mổ xẻ từng đoạn

```python
result = []

def dfs(node):
    if node is None:                    # (1) cơ sở
        return
    if node.left and not node.right:    # (2) cha quyết định
        result.append(node.left.val)
    if node.right and not node.left:
        result.append(node.right.val)
    dfs(node.left)                      # (3) đi xuống hai nhánh
    dfs(node.right)

dfs(root)
return result
```

- `node.left` dùng trong `if` được hiểu như `node.left is not None` (đối tượng `TreeNode` là "truthy", `None` là "falsy").
- `dfs` là **hàm lồng** (nested function): nó nhìn thấy biến `result` của hàm ngoài và gọi `result.append(...)` để thêm vào — không cần `global`/`nonlocal` vì ta **sửa** list chứ không **gán lại** biến.
- Gốc tự động không bao giờ được thêm, vì ta chỉ thêm **con** của một nút.

In [8]:
def lonely_dfs(root):
    result = []
    def dfs(node):
        if node is None:
            return
        if node.left and not node.right:
            result.append(node.left.val)
        if node.right and not node.left:
            result.append(node.right.val)
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return result

print(lonely_dfs(build_tree(ex1)))   # [4]
print(lonely_dfs(build_tree(ex2)))   # [6, 2]
print(lonely_dfs(build_tree(ex3)))   # [77, 55, 33, 66, 44, 22]

[4]
[6, 2]
[77, 55, 33, 66, 44, 22]


Thứ tự kết quả DFS (tiền thứ tự: nút → trái → phải) trùng với thứ tự trong output mẫu của đề, nhưng đề chấp nhận **mọi thứ tự**.

## 6. TRACE call stack — ví dụ 2

Ta in mỗi lần gọi `dfs` với độ thụt lề theo **độ sâu** đệ quy, kèm quyết định tại nút đó.

| Lời gọi | Nút | trái / phải | Thêm vào `result` | `result` sau |
|---|---:|---|---|---|
| dfs(7) | 7 | 1 / 4 | – | [] |
| dfs(1) | 1 | 6 / None | **6** | [6] |
| dfs(6) | 6 | None / None | – | [6] |
| dfs(4) | 4 | 5 / 3 | – | [6] |
| dfs(5) | 5 | None / None | – | [6] |
| dfs(3) | 3 | None / 2 | **2** | [6, 2] |
| dfs(2) | 2 | None / None | – | [6, 2] |

(Các lời gọi `dfs(None)` trả về ngay nên không ghi trong bảng.)

In [9]:
def trace_lonely_dfs(root):
    result = []
    def dfs(node, depth):
        if node is None:
            return
        pad = "  " * depth
        L = node.left.val if node.left else None
        R = node.right.val if node.right else None
        added = None
        if node.left and not node.right:
            added = node.left.val
        if node.right and not node.left:
            added = node.right.val
        if added is not None:
            result.append(added)
        print(f"{pad}dfs({node.val}) trái={L} phải={R}" + (f" → thêm {added}, result={result}" if added is not None else ""))
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return result

print("Kết quả:", trace_lonely_dfs(build_tree(ex2)))

dfs(7) trái=1 phải=4
  dfs(1) trái=6 phải=None → thêm 6, result=[6]
    dfs(6) trái=None phải=None
  dfs(4) trái=5 phải=3
    dfs(5) trái=None phải=None
    dfs(3) trái=None phải=2 → thêm 2, result=[6, 2]
      dfs(2) trái=None phải=None
Kết quả: [6, 2]


Ở trên ta gộp hai `if` thành một biến `added` vì một nút không thể vừa "chỉ có trái" vừa "chỉ có phải".

## 7. Cách 3 — DFS bằng stack (không đệ quy)

Đệ quy dùng **call stack** của Python; ta có thể tự quản lý một `stack` (list với `append`/`pop` ở cuối). Ích lợi: không lo giới hạn độ sâu đệ quy (mặc định khoảng 1000 tầng) khi cây rất "lệch".

In [10]:
def lonely_stack(root):
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        if node.left and not node.right:
            result.append(node.left.val)
        if node.right and not node.left:
            result.append(node.right.val)
        if node.right:
            stack.append(node.right)     # đẩy phải trước để trái được lấy ra trước
        if node.left:
            stack.append(node.left)
    return result

print(lonely_stack(build_tree(ex1)), lonely_stack(build_tree(ex2)), lonely_stack(build_tree(ex3)))

[4] [6, 2] [77, 55, 33, 66, 44, 22]


## 8. Cách 4 — BFS theo tầng bằng `deque`

Thay stack bằng **hàng đợi**: duyệt theo từng tầng. Kết quả theo thứ tự tầng (khác DFS) — vẫn hợp lệ.

In [11]:
def lonely_bfs(root):
    result = []
    q = deque([root])
    while q:
        node = q.popleft()
        if node.left and not node.right:
            result.append(node.left.val)
        if node.right and not node.left:
            result.append(node.right.val)
        for child in (node.left, node.right):
            if child:
                q.append(child)
    return result

print(lonely_bfs(build_tree(ex3)), "vs DFS:", lonely_dfs(build_tree(ex3)))

[77, 66, 55, 44, 33, 22] vs DFS: [77, 55, 33, 66, 44, 22]


In [12]:
def trace_lonely_bfs(root):
    result, q = [], deque([root])
    while q:
        print("hàng đợi:", [n.val for n in q])
        node = q.popleft()
        if (node.left is None) != (node.right is None):
            only = node.left or node.right
            result.append(only.val)
            print(f"  lấy {node.val}: chỉ có một con {only.val} → result={result}")
        for child in (node.left, node.right):
            if child:
                q.append(child)
    return result

trace_lonely_bfs(build_tree(ex1))

hàng đợi: [1]
hàng đợi: [2, 3]
  lấy 2: chỉ có một con 4 → result=[4]
hàng đợi: [3, 4]
hàng đợi: [4]


[4]

`node.left or node.right` trả **toán hạng truthy đầu tiên** — khi chỉ có một con, nó trả đúng con đó. Đây là cách gọn để lấy "con duy nhất".

## 9. Cách 5 — Đệ quy trả về list (kiểu hàm, không biến ngoài)

Mỗi lời gọi trả list nút cô đơn **trong cây con** của mình; ghép kết quả hai bên với phần của nút hiện tại.

In [13]:
def lonely_functional(node):
    if node is None:
        return []
    here = []
    if (node.left is None) != (node.right is None):
        here.append((node.left or node.right).val)
    return here + lonely_functional(node.left) + lonely_functional(node.right)

print(lonely_functional(build_tree(ex2)), lonely_functional(build_tree(ex3)))

[6, 2] [77, 55, 33, 66, 44, 22]


Gọn và đẹp, nhưng phép nối list `+` tạo list mới ở mỗi tầng → tệ nhất `O(n · h)` thời gian (với `h` = chiều cao). Bản `append` vào một list chung (cách 2) là `O(n)`.

## 10. Code nộp LeetCode (Python 3)

Trên LeetCode, `TreeNode`, `Optional`, `List` đã có sẵn; ở đây ta định nghĩa ở mục 2 để notebook chạy độc lập.

In [14]:
class Solution:
    def getLonelyNodes(self, root: Optional[TreeNode]) -> List[int]:
        result = []
        def dfs(node):
            if node is None:
                return
            if node.left and not node.right:
                result.append(node.left.val)
            if node.right and not node.left:
                result.append(node.right.val)
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return result

sol = Solution()
print(sol.getLonelyNodes(build_tree(ex1)))
print(sol.getLonelyNodes(build_tree(ex2)))
print(sol.getLonelyNodes(build_tree(ex3)))

[4]
[6, 2]
[77, 55, 33, 66, 44, 22]


Bản BFS (không đệ quy) cũng nộp được:

```python
class Solution:
    def getLonelyNodes(self, root: Optional[TreeNode]) -> List[int]:
        result, q = [], deque([root])
        while q:
            node = q.popleft()
            if (node.left is None) != (node.right is None):
                result.append((node.left or node.right).val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        return result
```

## 11. Vì sao thuật toán đúng?

- **Mỗi nút không phải gốc có đúng một cha.** DFS thăm **mọi** nút đúng một lần (mỗi nút được gọi `dfs` từ cha của nó, gốc được gọi từ ngoài).
- Tại cha `p`, ta thêm con `c` ⇔ `c` là con duy nhất của `p` ⇔ `c` cô đơn (đúng định nghĩa).
- Vì mỗi nút `c` chỉ có một cha, nó được xét **đúng một lần** → không bị thêm trùng, không bị sót.
- Gốc không có cha → không bao giờ được thêm. ✓

**Invariant (DFS):** khi `dfs(node)` kết thúc, `result` đã chứa đúng các nút cô đơn có cha nằm trong cây con của `node` (cộng những gì có từ trước).

## 12. Phân tích Time / Space

Gọi `n` = số nút, `h` = chiều cao cây (`log n` nếu cân bằng, tới `n` nếu cây lệch thành "đường thẳng").

| Cách | Time | Space phụ (không tính output) | Ghi chú |
|---|---|---|---|
| 1. Bảng `parent` | `O(n)` | `O(n)` | hai lượt, dict + list |
| 2. DFS đệ quy | `O(n)` | `O(h)` call stack | **khuyên dùng** |
| 3. DFS stack | `O(n)` | `O(h)` | không lo giới hạn đệ quy |
| 4. BFS `deque` | `O(n)` | `O(w)` với `w` = độ rộng lớn nhất | thứ tự theo tầng |
| 5. Đệ quy trả list | `O(n · h)` tệ nhất | `O(n)` | do nối list |

Với `n <= 1000`, cây lệch có `h = 1000` — sát giới hạn đệ quy mặc định của Python (`sys.getrecursionlimit()` thường là 1000). Trên LeetCode giới hạn thường được nâng nên vẫn qua, nhưng biết cách stack/BFS là an toàn hơn (xem mục 13.2).

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Vì thứ tự tùy ý, ta so sánh **sau khi sắp xếp**.

In [15]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.getLonelyNodes(build_tree(ex1)), [4])
check("ví dụ 2", sol.getLonelyNodes(build_tree(ex2)), [6, 2])
check("ví dụ 3", sol.getLonelyNodes(build_tree(ex3)), [77, 55, 33, 66, 44, 22])
check("chỉ có gốc", sol.getLonelyNodes(build_tree([5])), [])
check("gốc + một con trái", sol.getLonelyNodes(build_tree([5, 8])), [8])
check("cây đầy đủ", sol.getLonelyNodes(build_tree([1, 2, 3, 4, 5, 6, 7])), [])
check("đường thẳng lệch phải", sol.getLonelyNodes(build_tree([1, None, 2, None, 3])), [2, 3])
check("giá trị trùng nhau", sol.getLonelyNodes(build_tree([1, 1, 1, 1, None, None, 1])), [1, 1])

PASS  ví dụ 1: [4]
PASS  ví dụ 2: [6, 2]
PASS  ví dụ 3: [77, 55, 33, 66, 44, 22]
PASS  chỉ có gốc: []
PASS  gốc + một con trái: [8]
PASS  cây đầy đủ: []
PASS  đường thẳng lệch phải: [2, 3]
PASS  giá trị trùng nhau: [1, 1]


### 13.1. Đối chiếu 5 cách trên cây ngẫu nhiên (có seed)

In [16]:
import random

def random_tree(rng, n):
    nodes = [TreeNode(rng.randint(1, 10**6))]
    while len(nodes) < n:
        parent = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(parent, side) is None:
            child = TreeNode(rng.randint(1, 10**6))
            setattr(parent, side, child)
            nodes.append(child)
    return nodes[0]

rng = random.Random(1469)
methods = [lonely_parent_map, lonely_dfs, lonely_stack, lonely_bfs, lonely_functional, Solution().getLonelyNodes]
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 200))
    results = {tuple(sorted(f(root))) for f in methods}
    assert len(results) == 1, results
print("300 cây ngẫu nhiên: 6 hàm cho cùng tập kết quả")

300 cây ngẫu nhiên: 6 hàm cho cùng tập kết quả


`getattr(obj, "left")` đọc thuộc tính theo **tên dạng chuỗi**, `setattr(obj, "left", v)` gán — tiện khi chọn ngẫu nhiên trái/phải.

### 13.2. Cây lệch 1000 nút — giới hạn đệ quy

In [17]:
import sys
deep = TreeNode(1)
cur = deep
for v in range(2, 1001):
    cur.right = TreeNode(v)
    cur = cur.right
print("recursion limit:", sys.getrecursionlimit())
print("stack:", len(lonely_stack(deep)), "| bfs:", len(lonely_bfs(deep)))
try:
    print("dfs đệ quy:", len(lonely_dfs(deep)))
except RecursionError as e:
    print("dfs đệ quy gặp RecursionError →", type(e).__name__)

recursion limit: 1000
stack: 999 | bfs: 999
dfs đệ quy gặp RecursionError → RecursionError


Tùy môi trường, bản đệ quy có thể chạy được hoặc gặp `RecursionError` ở đúng ngưỡng ~1000 tầng (call stack còn chứa cả các hàm của Jupyter). Cách stack/BFS luôn an toàn. Có thể nâng giới hạn bằng `sys.setrecursionlimit(...)` nhưng nên hiểu rủi ro.

## 14. Lỗi người mới hay gặp

### Lỗi 1 — Đứng ở nút con và coi "lá" là cô đơn

In [18]:
def wrong_leaves(root):
    result = []
    def dfs(node):
        if node is None:
            return
        if node.left is None and node.right is None:   # lá ≠ cô đơn!
            result.append(node.val)
        dfs(node.left); dfs(node.right)
    dfs(root)
    return result

print("Sai:", wrong_leaves(build_tree(ex1)), "| Đúng:", lonely_dfs(build_tree(ex1)))

Sai: [4, 3] | Đúng: [4]


**Vì sao sai:** "cô đơn" nói về **anh em** (cha có mấy con), không phải nút có con hay không.

### Lỗi 2 — Thêm cả gốc

In [19]:
def wrong_include_root(root):
    return [root.val] + lonely_dfs(root)

print("Sai:", wrong_include_root(build_tree([5, 8])), "| Đúng:", lonely_dfs(build_tree([5, 8])))

Sai: [5, 8] | Đúng: [8]


### Lỗi 3 — Dùng `or` thay vì kiểm tra "đúng một con"

```python
if node.left or node.right:      # đúng cả khi có HAI con
```

In [20]:
def wrong_or(root):
    result = []
    def dfs(node):
        if node is None:
            return
        if node.left or node.right:
            result.append((node.left or node.right).val)
        dfs(node.left); dfs(node.right)
    dfs(root)
    return result

print("Sai:", wrong_or(build_tree([1, 2, 3])), "| Đúng:", lonely_dfs(build_tree([1, 2, 3])))

Sai: [2] | Đúng: []


**Sửa:** `(node.left is None) != (node.right is None)`.

### Lỗi 4 — Truy cập `node.left.val` khi `node.left` là `None`

In [21]:
try:
    leaf = TreeNode(9)
    print(leaf.left.val)
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'NoneType' object has no attribute 'val'


**Sửa:** luôn kiểm tra `node.left` khác `None` trước khi đọc `.val` — trong lời giải, phép kiểm tra nằm trong cùng điều kiện `if node.left and not node.right`.

### Lỗi 5 — Gán lại biến ngoài trong hàm lồng

```python
count = 0
def dfs(node):
    count += 1      # UnboundLocalError: gán lại biến của hàm ngoài
```
`result.append(...)` thì **không** lỗi vì chỉ sửa đối tượng. Nếu cần gán lại số nguyên, dùng `nonlocal count`.

In [22]:
def count_lonely(root):
    count = 0
    def dfs(node):
        nonlocal count
        if node is None:
            return
        if (node.left is None) != (node.right is None):
            count += 1
        dfs(node.left); dfs(node.right)
    dfs(root)
    return count

print(count_lonely(build_tree(ex3)))   # 6

6


## 15. Mẹo nhận dạng pattern

- Thuộc tính nói về **quan hệ cha–con / anh em** → xử lý tại **nút cha**, nơi nhìn thấy cả hai con.
- "Duyệt mọi nút, thu thập theo điều kiện cục bộ" → DFS (đệ quy hoặc stack) hoặc BFS đều được; chọn đệ quy cho gọn, stack/BFS khi cây có thể rất sâu.
- Kiểm tra "đúng một trong hai" → XOR trên `bool`: `a != b`.
- Bài cùng họ: *Sum of Left Leaves* (404), *Cousins in Binary Tree* (993), *Count Univalue Subtrees* (250), *Binary Tree Paths* (257).

## 16. Biến thể

### 16.1. Đếm nút lá, nút có một con, nút có hai con

In [23]:
def degree_stats(root):
    stats = {0: 0, 1: 0, 2: 0}
    stack = [root]
    while stack:
        node = stack.pop()
        k = (node.left is not None) + (node.right is not None)
        stats[k] += 1
        stack.extend(c for c in (node.left, node.right) if c)
    return stats

print(degree_stats(build_tree(ex2)))   # lá / một con / hai con

{0: 3, 1: 2, 2: 2}


Nhận xét: số nút cô đơn **bằng** số nút có đúng một con (mỗi nút như vậy sinh ra đúng một con cô đơn).

### 16.2. Sum of Left Leaves (LeetCode 404) — cũng quyết định tại nút cha

In [24]:
def sum_of_left_leaves(root):
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        L = node.left
        if L and L.left is None and L.right is None:
            total += L.val
        stack.extend(c for c in (node.left, node.right) if c)
    return total

print(sum_of_left_leaves(build_tree([3, 9, 20, None, None, 15, 7])))   # 24

24


### 16.3. Nút cô đơn theo từng tầng

In [25]:
def lonely_by_level(root):
    levels, q, depth = {}, deque([(root, 0)]), 0
    while q:
        node, d = q.popleft()
        if (node.left is None) != (node.right is None):
            levels.setdefault(d + 1, []).append((node.left or node.right).val)
        for c in (node.left, node.right):
            if c:
                q.append((c, d + 1))
    return levels

print(lonely_by_level(build_tree(ex3)))

{2: [77, 66], 3: [55, 44], 4: [33, 22]}


`dict.setdefault(key, [])` trả list của `key`, tạo list rỗng nếu chưa có — tiện để gom nhóm.

## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Vẽ tay.** Vẽ cây `[1, 2, None, 3, 4, None, 5]` và liệt kê nút cô đơn.

**Bài 2 — Dự đoán output.**
```python
x = 5
print((None is None) != (x is None), None or 7, 3 or 7, 0 or None)
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết DFS đệ quy

In [26]:
def exercise_lonely(root):
    # TODO 1: result = []
    # TODO 2: hàm dfs(node): None -> return; nếu đúng một con -> thêm con đó; gọi dfs hai bên
    # TODO 3: dfs(root); return result
    pass

# assert sorted(exercise_lonely(build_tree([7,1,4,6,None,5,3,None,None,None,None,None,2]))) == [2, 6]

### Bài 4 — TODO: tổng giá trị các nút cô đơn, dùng BFS

In [27]:
def exercise_sum_lonely(root):
    # TODO: deque, tại mỗi nút kiểm tra XOR, cộng giá trị con duy nhất
    pass

# assert exercise_sum_lonely(build_tree([11,99,88,77,None,None,66,55,None,None,44,33,None,None,22])) == 297

### Bài 5 — TODO: Cousins in Binary Tree (LeetCode 993)
Giá trị các nút phân biệt. Hai nút `x`, `y` là **anh em họ** nếu cùng độ sâu nhưng **khác cha**.

In [28]:
def exercise_is_cousins(root, x, y):
    # TODO: BFS lưu (node, parent, depth); tìm thông tin của x và y rồi so sánh
    pass

# assert exercise_is_cousins(build_tree([1,2,3,4]), 4, 3) == False
# assert exercise_is_cousins(build_tree([1,2,3,None,4,None,5]), 5, 4) == True
# assert exercise_is_cousins(build_tree([1,2,3,None,4]), 2, 3) == False

### Bài 6 — Mini challenge: cắt mọi nút cô đơn
Trả về **số nút còn lại** nếu ta xóa mọi nút cô đơn (xóa nút thì xóa luôn cả cây con của nó). Gốc luôn được giữ.

In [29]:
def exercise_remaining_after_cut(root):
    # TODO: DFS; tại nút có đúng một con thì KHÔNG đi xuống con đó (cả cây con bị xóa)
    pass

# assert exercise_remaining_after_cut(build_tree([1,2,3,None,4])) == 3
# assert exercise_remaining_after_cut(build_tree([1,None,2,None,3])) == 1

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Cha quyết định. | hàm lồng `dfs` + list chung. | `if (node.left is None) != (node.right is None): result.append((node.left or node.right).val)` |
| 4 | Giống bài 3, đổi `append` thành cộng. | `deque([root])`, `popleft`. | `total += (node.left or node.right).val` |
| 5 | Cần độ sâu **và** cha của từng nút. | BFS phần tử `(node, parent, depth)`. | `info[node.val] = (parent, depth)`; so `dx == dy and px is not py` |
| 6 | Nút có hai con: đi tiếp cả hai; có một con: dừng nhánh đó. | DFS đếm. | `if k == 2: return 1 + f(L) + f(R)`; `return 1` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** cây: `1` chỉ có con trái `2`; `2` có hai con `3`, `4`; theo level-order, cặp kế tiếp `None, 5` là con của `3` → `3` chỉ có con phải `5`. Nút cô đơn: **2** (con một của 1) và **5** (con một của 3).
**Bài 2:** `True 7 3 None`.

In [30]:
def answer_lonely(root):
    result = []
    def dfs(node):
        if node is None:
            return
        if (node.left is None) != (node.right is None):
            result.append((node.left or node.right).val)
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return result

def answer_sum_lonely(root):
    total, q = 0, deque([root])
    while q:
        node = q.popleft()
        if (node.left is None) != (node.right is None):
            total += (node.left or node.right).val
        for c in (node.left, node.right):
            if c:
                q.append(c)
    return total

def answer_is_cousins(root, x, y):
    info, q = {}, deque([(root, None, 0)])
    while q:
        node, parent, depth = q.popleft()
        info[node.val] = (parent, depth)
        for c in (node.left, node.right):
            if c:
                q.append((c, node, depth + 1))
    (px, dx), (py, dy) = info[x], info[y]
    return dx == dy and px is not py

def answer_remaining_after_cut(root):
    def f(node):
        k = (node.left is not None) + (node.right is not None)
        if k == 2:
            return 1 + f(node.left) + f(node.right)
        return 1          # lá, hoặc con duy nhất bị cắt cùng cây con
    return f(root)

assert sorted(answer_lonely(build_tree([7,1,4,6,None,5,3,None,None,None,None,None,2]))) == [2, 6]
assert answer_sum_lonely(build_tree([11,99,88,77,None,None,66,55,None,None,44,33,None,None,22])) == 297
assert answer_is_cousins(build_tree([1,2,3,4]), 4, 3) == False
assert answer_is_cousins(build_tree([1,2,3,None,4,None,5]), 5, 4) == True
assert answer_is_cousins(build_tree([1,2,3,None,4]), 2, 3) == False
assert answer_remaining_after_cut(build_tree([1,2,3,None,4])) == 3
assert answer_remaining_after_cut(build_tree([1,None,2,None,3])) == 1
print("Bài 1:", answer_lonely(build_tree([1, 2, None, 3, 4, None, 5])))
print_tree(build_tree([1, 2, None, 3, 4, None, 5]))
x = 5
print("Bài 2:", (None is None) != (x is None), None or 7, 3 or 7, 0 or None)
print("Đáp án bài 3–6: PASS")

Bài 1: [2, 5]
1
            4
      2
                  5
            3
Bài 2: True 7 3 None
Đáp án bài 3–6: PASS


## 20. Template nhớ nhanh + checklist submit

```python
result = []
def dfs(node):
    if node is None:
        return
    if (node.left is None) != (node.right is None):   # đúng MỘT con
        result.append((node.left or node.right).val)  # con đó cô đơn
    dfs(node.left)
    dfs(node.right)
dfs(root)
return result
```

**Checklist trước khi nộp**
- [ ] Quyết định tại **nút cha** (nhìn cả hai con).
- [ ] "Đúng một con" = XOR, không phải `or`.
- [ ] Không thêm gốc; không nhầm "lá" với "cô đơn".
- [ ] Kiểm tra `None` trước khi đọc `.val`.
- [ ] Thứ tự kết quả tùy ý — test bằng `sorted`.
- [ ] Method tên `getLonelyNodes`, trả về `List[int]`.